# 13.5 — Self-supervised audio (wav2vec, HuBERT)

Self-supervised audio learns speech features by hiding or contrasting parts of the signal before any transcript is available. This notebook uses synthetic NumPy audio only: masked frames, positives, and easy/hard negatives stand in for wav2vec-style and HuBERT-style pretraining. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 1305
rng = np.random.default_rng(SEED)
FS = 8000


def sine(freq, seconds=0.35, fs=FS, phase=0.0, amp=1.0):
    n = int(seconds * fs)
    t = np.arange(n) / fs
    return amp * np.sin(2 * np.pi * freq * t + phase)


def mix_tones(freqs, seconds=0.35, fs=FS, amps=None):
    if amps is None:
        amps = np.ones(len(freqs))
    x = np.zeros(int(seconds * fs))
    for freq, amp in zip(freqs, amps):
        x = x + sine(freq, seconds, fs, amp=amp)
    scale = np.max(np.abs(x)) + 1e-12
    return x / scale


def chirp(start, stop, seconds=0.6, fs=FS):
    n = int(seconds * fs)
    t = np.arange(n) / fs
    k = (stop - start) / max(seconds, 1e-12)
    phase = 2 * np.pi * (start * t + 0.5 * k * t * t)
    return np.sin(phase)


def add_noise(x, scale, local_rng=rng):
    return x + local_rng.normal(0.0, scale, size=len(x))


def stft_mag(x, n_fft=256, hop=96):
    if len(x) < n_fft:
        x = np.pad(x, (0, n_fft - len(x)))
    frames = []
    window = np.hanning(n_fft)
    for start in range(0, len(x) - n_fft + 1, hop):
        frame = x[start:start + n_fft]
        spec = np.abs(np.fft.rfft(frame * window))
        frames.append(spec)
    if not frames:
        frames.append(np.abs(np.fft.rfft(x[:n_fft] * window)))
    return np.array(frames)


def frame_audio(x, frame_size=256, hop=128):
    frames = []
    for start in range(0, len(x) - frame_size + 1, hop):
        frames.append(x[start:start + frame_size])
    if not frames:
        frames.append(np.pad(x, (0, max(0, frame_size - len(x))))[:frame_size])
    return np.array(frames)


def normalize_rows(values):
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.maximum(norms, 1e-12)


def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / max(denom, 1e-12))


def softmax(logits):
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)


def top_frequency(x, fs=FS, n_fft=512):
    windowed = x[:n_fft]
    if len(windowed) < n_fft:
        windowed = np.pad(windowed, (0, n_fft - len(windowed)))
    mag = np.abs(np.fft.rfft(windowed * np.hanning(n_fft)))
    freqs = np.fft.rfftfreq(n_fft, 1 / fs)
    return float(freqs[int(np.argmax(mag))])


def ladder_summary(rungs):
    for rung in rungs:
        x = rung["audio"]
        label = rung.get("label", rung.get("target", "synthetic"))
        print(rung["name"], x.shape, label)



def contrastive_audio_score(context, positive, negatives, tau=0.1):
    candidates = [positive] + list(negatives)
    sims = np.array([cosine(context, item) for item in candidates])
    probs = softmax(sims / tau)
    loss = -np.log(probs[0] + 1e-12)
    return probs, sims, float(loss)


def masked_cluster_loss(probs, targets, mask):
    chosen = probs[np.arange(len(targets)), targets]
    masked = -np.log(chosen[mask] + 1e-12)
    return float(np.mean(masked))


def ssl_embedding(x):
    spec = stft_mag(x, n_fft=256, hop=96)
    log_spec = np.log1p(spec)
    centroid = np.sum(log_spec * np.arange(log_spec.shape[1]), axis=1)
    centroid = centroid / np.maximum(np.sum(log_spec, axis=1), 1e-12)
    stats = np.array([
        np.mean(log_spec),
        np.std(log_spec),
        np.mean(centroid),
        np.std(centroid),
    ])
    return stats


def make_ssl_ladder():
    d1 = sine(440, seconds=0.35)
    d2 = mix_tones([440, 660], seconds=0.40, amps=[1.0, 0.55])
    d3 = add_noise(d2, 0.08)
    d4 = np.concatenate([
        sine(330, seconds=0.20),
        chirp(380, 760, seconds=0.30),
        mix_tones([440, 880], seconds=0.25, amps=[1.0, 0.4]),
    ])
    d5 = np.concatenate([
        add_noise(chirp(300, 900, seconds=0.40), 0.12),
        add_noise(mix_tones([440, 450], seconds=0.45, amps=[1.0, 0.9]), 0.16),
        add_noise(sine(660, seconds=0.35), 0.14),
    ])
    return [
        {"name": "D1 sine", "audio": d1, "freq": 440, "hard_freq": 445},
        {"name": "D2 two-tone", "audio": d2, "freq": 440, "hard_freq": 660},
        {"name": "D3 +noise", "audio": d3, "freq": 440, "hard_freq": 455},
        {"name": "D4 multi-segment", "audio": d4, "freq": 440, "hard_freq": 760},
        {"name": "D5 longer noisy hard negatives", "audio": d5, "freq": 440, "hard_freq": 450},
    ]


def evaluate_ssl_rung(rung):
    x = rung["audio"]
    context = ssl_embedding(x)
    positive = ssl_embedding(add_noise(x, 0.02))
    easy_neg = ssl_embedding(sine(1200, seconds=len(x) / FS))
    hard_neg = ssl_embedding(add_noise(sine(rung["hard_freq"], seconds=len(x) / FS), 0.10))
    negatives = [easy_neg, hard_neg]
    probs, sims, loss = contrastive_audio_score(context, positive, negatives, tau=0.1)
    prediction = int(np.argmax(probs))
    return {
        "accuracy": float(prediction == 0),
        "loss": loss,
        "similarities": sims,
        "probabilities": probs,
        "spectrogram": stft_mag(x),
    }


## The concept, built once (D1): contrastive prediction

The wav2vec-style decision is $\mathcal{L}=-\lograc{\exp(\operatorname{sim}(c_t,q^+)/	au)}{\exp(\operatorname{sim}(c_t,q^+)/	au)+\sum_j\exp(\operatorname{sim}(c_t,q^-_j)/	au)}$. With lesson vectors the cosine scores are $0.800$ and $0.100$ at $	au=0.1$.

In [ ]:

context = np.array([1.0, 0.0])
positive = np.array([0.8, 0.6])
negative = np.array([0.1, 0.995])
probs, sims, loss = contrastive_audio_score(context, positive, [negative], tau=0.1)
print("cosines", np.round(sims, 3))
print("positive probability", round(float(probs[0]), 4))
print("loss", round(loss, 4))
assert np.allclose(np.round(sims, 3), [0.800, 0.100])
assert round(float(probs[0]), 4) == 0.9991
assert round(loss, 4) == 0.0009


## Masked cluster variant

HuBERT-like training hides frames and predicts bootstrap clusters with cross-entropy. If $3$ of $10$ frames are masked, the mask fraction is $3/10=0.300$; if the target cluster probability is $0.7$, the loss is $-\log(0.7)=0.357$.

In [ ]:

mask = np.array([True, False, True, False, False, True, False, False, False, False])
mask_fraction = float(np.mean(mask))
cluster_probs = np.array([[0.1, 0.2, 0.7]])
cluster_loss = -np.log(cluster_probs[0, 2])
print("mask fraction", round(mask_fraction, 3))
print("cluster loss", round(float(cluster_loss), 3))
assert round(mask_fraction, 3) == 0.300
assert round(float(cluster_loss), 3) == 0.357


## The dataset ladder

F7 audio ladder inline: sine $ightarrow$ two-tone $ightarrow$ plus noise $ightarrow$ multi-segment $ightarrow$ longer and noisier with hard negatives.

In [ ]:

rungs = make_ssl_ladder()
ladder_summary(rungs)
print("sample", np.round(rungs[0]["audio"][:8], 3))


In [ ]:

results = []
for rung in rungs:
    result = evaluate_ssl_rung(rung)
    results.append(result)
    print(rung["name"], "top1", result["accuracy"], "loss", round(result["loss"], 4))
metrics = np.array([item["accuracy"] for item in results])


## Results visualization

The closing figure has one artifact panel per D1–D5 rung plus a metric curve over the same ladder.

In [ ]:

fig, axes = plt.subplots(2, len(rungs), figsize=(15, 5))
for idx, rung in enumerate(rungs):
    spec = results[idx]["spectrogram"]
    axes[0, idx].imshow(np.log1p(spec).T, origin="lower", aspect="auto")
    axes[0, idx].set_title(rung["name"])
    sims = results[idx]["similarities"]
    axes[1, idx].bar(np.arange(len(sims)), sims)
    axes[1, idx].set_xticks([0, 1, 2])
    axes[1, idx].set_xticklabels(["pos", "easy", "hard"])
fig.suptitle("Masked spectrogram panels and positive-negative similarity bars")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(np.arange(1, 6), metrics, marker="o")
plt.xticks(np.arange(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(-0.05, 1.05)
plt.ylabel("contrastive top-1 accuracy")
plt.title("Top-1 vs complexity")
plt.show()


## Pitfall on D5: negatives that are too easy

Easy negatives can make the loss tiny without forcing acoustic detail. The fix is to include same-frequency/noisy hard negatives and keep a calibrated mask rate.

In [ ]:

d5 = rungs[-1]
context = ssl_embedding(d5["audio"])
positive = ssl_embedding(add_noise(d5["audio"], 0.02))
easy_negatives = [ssl_embedding(sine(freq, seconds=len(d5["audio"]) / FS)) for freq in [1200, 1500, 1800]]
hard_negatives = [ssl_embedding(add_noise(sine(freq, seconds=len(d5["audio"]) / FS), 0.12)) for freq in [440, 445, 450]]
easy_probs, easy_sims, easy_loss = contrastive_audio_score(context, positive, easy_negatives, tau=0.1)
hard_probs, hard_sims, hard_loss = contrastive_audio_score(context, positive, hard_negatives, tau=0.1)
print("easy negative loss", round(easy_loss, 6), "top1", int(np.argmax(easy_probs) == 0))
print("hard negative loss", round(hard_loss, 6), "top1", int(np.argmax(hard_probs) == 0))
print("calibrated mask rate", 0.20)


## Evaluate it + Practice

- Metric: contrastive top-1 accuracy; no-skill baseline is $1/(1+K)$ for $K$ negatives.
- Sanity check: the positive augmentation should be closest on D1.
- Ablation: remove hard negatives and the loss can look deceptively good.
- Failure signals: all similarities equal, mask fraction near $0$ or $1$, or D5 top-1 collapsing.

Practice 1: change the temperature and plot the probability of the positive.


Practice 2: increase the D5 hard-negative frequency gap and explain the loss change.